# 文本嵌入模型精度测评报告 (Embedding Model Benchmark Report)

本 Notebook 用于对多款主流文本嵌入模型进行标准化测评。重点考察各模型在处理 **CS 论文学术语料** 时的语义召回精度与排名表现。

In [ ]:
!pip install -qU sentence-transformers transformers torch datasets pandas beir matplotlib

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from typing import List
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

print(f"CUDA available: {torch.cuda.is_available()}")

## 1. 加载学术评测集 (SciFact)
SciFact 包含真实人类标注的科学论文事实核查数据，是评估学术语料检索能力的权威数据集。

In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import random

dataset = "scifact"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
out_dir = os.path.join(os.getcwd(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

# 随机抽样 100 个 Query 和 1000 篇论文作为搜索池以保证评测效率
SAMPLE_QUERIES = 100
sampled_q_ids = random.sample(list(qrels.keys()), min(SAMPLE_QUERIES, len(qrels)))

test_queries = {qid: queries[qid] for qid in sampled_q_ids}
relevant_docs = {qid: [did for did, score in qrels[qid].items() if score > 0] for qid in sampled_q_ids}

all_target_docs = set()
for docs in relevant_docs.values(): all_target_docs.update(docs)

filler_docs = random.sample(list(corpus.keys()), min(1000, len(corpus)))
final_corpus_ids = list(all_target_docs.union(set(filler_docs)))
test_corpus = {did: f"Title: {corpus[did]['title']}\nAbstract: {corpus[did]['text']}" for did in final_corpus_ids}

print(f"Current Setup: {len(test_queries)} Queries, {len(test_corpus)} Documents")

## 2. 定义统一评测函数

In [ ]:
def evaluate_model(model_name: str, query_prefix: str = "", doc_prefix: str = ""):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n>>> Processing: {model_name} on {device}")
    
    try:
        model = SentenceTransformer(model_name, device=device)
    except Exception as e:
        print(f"  [!] Model load failed: {e}. Attempting basic load...")
        # For some models that are not strict sentence-transformers
        model = SentenceTransformer(model_name, device=device, trust_remote_code=True)

    model.eval()
    
    start_time = time.time()
    
    c_ids = list(test_corpus.keys())
    c_texts = [doc_prefix + str(t)[:8000] for t in test_corpus.values()]
    corpus_embeddings = model.encode(c_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=32)
    
    q_ids = list(test_queries.keys())
    q_texts = [query_prefix + t for t in test_queries.values()]
    query_embeddings = model.encode(q_texts, normalize_embeddings=True, show_progress_bar=False)
    
    sim_matrix = np.dot(query_embeddings, corpus_embeddings.T)
    
    ndcg_10 = 0.0
    hit_3 = 0.0
    
    for i, qid in enumerate(q_ids):
        scores = sim_matrix[i]
        top_indices = np.argsort(scores)[::-1][:20]
        top_doc_ids = [c_ids[idx] for idx in top_indices]
        
        targets = set(relevant_docs[qid])
        
        dcg = sum(1.0 / np.log2(rank + 2) for rank, did in enumerate(top_doc_ids[:10]) if did in targets)
        idcg = sum(1.0 / np.log2(rank + 2) for rank in range(min(len(targets), 10)))
        ndcg_10 += (dcg / idcg if idcg > 0 else 0)
        
        if any(did in targets for did in top_doc_ids[:3]): hit_3 += 1.0
            
    return {
        'Model': model_name,
        'NDCG@10': round(ndcg_10 / len(q_ids) * 100, 2),
        'Hit@3 (%)': round(hit_3 / len(q_ids) * 100, 2),
        'Time (s)': round(time.time() - start_time, 1),
        'Dim': corpus_embeddings.shape[1]
    }

## 3. 执行对比评测

In [ ]:
model_configs = [
    {'name': 'BAAI/bge-m3', 'q_p': '', 'd_p': ''},
    {'name': 'nomic-ai/nomic-embed-text-v1.5', 'q_p': 'search_query: ', 'd_p': 'search_document: '},
    {'name': 'Alibaba-NLP/gte-large-en-v1.5', 'q_p': '', 'd_p': ''},
    {'name': 'allenai/specter2_base', 'q_p': '', 'd_p': ''}, # 第一代 Specter 的官方继任者
    {'name': 'allenai-specter', 'q_p': '', 'd_p': ''},       # 改用兼容 ST 的名称
    {'name': 'sentence-transformers/allenai-specter', 'q_p': '', 'd_p': ''},
    {'name': 'allenai/scibert_scivocab_uncased', 'q_p': '', 'd_p': ''} # 经典的学术模型
]

results_list = []
for cfg in model_configs:
    try:
        res = evaluate_model(cfg['name'], cfg['q_p'], cfg['d_p'])
        results_list.append(res)
    except Exception as e:
        print(f"Error testing {cfg['name']}: {e}")

df = pd.DataFrame(results_list).sort_values('NDCG@10', ascending=False)
display(df)

## 4. 评测结果可视化

In [ ]:
if not df.empty:
    plt.figure(figsize=(12, 6))
    colors = plt.cm.get_cmap('viridis', len(df))
    bars = plt.bar(df['Model'], df['NDCG@10'], color=colors.colors)
    plt.xlabel('Model')
    plt.ylabel('NDCG@10 Score')
    plt.title('Accuracy Comparison (NDCG@10)')
    plt.xticks(rotation=45, ha='right')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, yval, ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.show()